# Dataset Construction

- Align trades to most recent LOB snapshot (no lookahead)
- Flatten LOB into tabular format (depth levels)
- Aggregate trades per timestamp
- Build microstructure features using only time t data
- Create targets:
  - move = (mid_price_change_1 != 0)
  - direction = sign(mid_price_change_1) where move = 1

Output is a single dataset indexed by timestamp

### Batch Pipeline

- LOB sampled every second - one row per timestamp
- Trades grouped within each second
- Trades aligned to latest LOB snapshot at or before trade time

Trades within a second are matched to that second’s LOB snapshot

Output:
- LOB: one row per timestamp
- Trades: aggregated per timestamp and aligned to LOB

In [1]:
import glob
from pathlib import Path
import gc
from tqdm import tqdm
import glob
import pandas as pd
import numpy as np

from microstructure_alpha.utils.logger import setup_logger
from microstructure_alpha.data.pipeline import run_batch_build_dataset_pipeline

In [2]:
logger = setup_logger(
    "C:\\Users\\jayod\\Documents\\Quant_Project\\microstructure-alpha-engine\\logs\\02_building_dataset_notebook.log"
)

In [3]:
trade_path = "C:\\Users\\jayod\\Documents\\Quant_Project\\microstructure-alpha-engine\\data\\raw_26_03_36\\trades\\"
lob_path = "C:\\Users\\jayod\\Documents\\Quant_Project\\microstructure-alpha-engine\\data\\raw_26_03_36\\lob\\"
save_path = "C:\\Users\\jayod\\Documents\\Quant_Project\\microstructure-alpha-engine\\data\\interim\\"

RUN_PIPELINE = True
if RUN_PIPELINE:
    run_batch_build_dataset_pipeline(lob_path, trade_path, save_path)

2026-04-27 15:28:20 | INFO | Starting batch pipeline
2026-04-27 15:28:20 | INFO | Found 95 trade files
2026-04-27 15:28:20 | INFO | Found 95 lob files
2026-04-27 15:28:20 | INFO | Processing 95 batches


Processing batches: 100%|██████████| 95/95 [00:08<00:00, 11.04it/s]

2026-04-27 15:28:29 | INFO | Batch pipeline completed successfully


## Feature Engineering

Build microstructure features from LOB and trade data using only information available at time t

### Features
- Price and spread  
- Liquidity  
- Imbalance  
- Microprice  
- Trade flow  
- Volatility  
- Momentum  

### Transformations
- Log / log1p for skewed features

### Output

Final dataset indexed by timestamp representing market state at time t, used to predict t+1 price movement

In [4]:
from microstructure_alpha.data.loader import load_parquet_glob
from microstructure_alpha.features.lob_features import build_lob_feature_pipeline
from microstructure_alpha.features.trade_features import build_trade_feature_pipeline
from microstructure_alpha.features.merge_features import merge_lob_and_trade

In [5]:
lob_df = load_parquet_glob(
    "C:\\Users\\jayod\\Documents\\Quant_Project\\microstructure-alpha-engine\\data\\interim\\lob_interim*",
    sort_by="timestamp",
)
trade_df = load_parquet_glob(
    "C:\\Users\\jayod\\Documents\\Quant_Project\\microstructure-alpha-engine\\data\\interim\\trade_interim*",
    sort_by="timestamp",
)

In [6]:
lob_with_features = build_lob_feature_pipeline(lob_df)
trade_with_features = build_trade_feature_pipeline(trade_df)
final_dataset = merge_lob_and_trade(trade_with_features, lob_with_features)

In [7]:
final_dataset = final_dataset.reset_index(drop=True)
trade_cols = [
    "trade_count",
    "buy_count",
    "sell_count",
    "total_trade_volume",
    "buy_volume",
    "sell_volume",
    "avg_trade_size",
    "max_trade_size",
    "min_trade_size",
    "std_trade_size",
    "vwap",
    "max_over_average",
    "trade_volume_imbalance",
    "trade_volume_change",
    "trade_count_change",
    "trade_count_imbalance",
    "lag_trade_volume_imbalance_1",
    "lag_trade_volume_imbalance_2",
    "lag_trade_volume_imbalance_3",
    "lag_trade_volume_imbalance_5",
]

final_dataset[trade_cols] = final_dataset[trade_cols].fillna(0)
final_dataset.dropna(inplace=True)

### Feature Transforms

- log / log1p applied to skewed features to stabilise scale and reduce impact of outliers

In [8]:
from microstructure_alpha.plots import plot_check_features_distr

Check representative features for each feature group (in microstructure_alpha.features.feature_lists.py)

In [9]:
check_features = [
    "lob_bids_volume_1",
    "lob_depth_ratio_5",
    "rel_spread",
    "imbalance_1",
    "total_book_volume",
    "mid_minus_micro",
    "log_return_1",
    "momentum_5_log_return_1",
    "vol_5",
    "trade_count",
    "total_trade_volume",
    "trade_volume_imbalance",
    "trade_volume_change",
    "lag_trade_volume_imbalance_1",
]

fig, axes = plot_check_features_distr(
    df=final_dataset,
    features=check_features,
    bins=100,
    n_cols=4,
    show=False,
    save_path="C:\\Users\\jayod\\Documents\\Quant_Project\\microstructure-alpha-engine\\notebooks\\notebook_figs\\02_figs\\var_transform_check.pdf",
)

In [10]:
from microstructure_alpha.features.feature_lists import FEATURES_TO_TRANSFORM
from microstructure_alpha.features.feature_transforms import (
    signed_log1p,
    apply_transform,
)

In [11]:
final_dataset = apply_transform(
    df=final_dataset,
    features=FEATURES_TO_TRANSFORM,
    func=signed_log1p,
    suffix="_log1p",
    use_values=True,
)

In [12]:
PATH = "C:\\Users\\jayod\\Documents\\Quant_Project\\microstructure-alpha-engine\\data\\processed\\final_dataset_40k_feature_transform.parquet"
final_dataset.to_parquet(
    PATH,
    compression="snappy",
)

### Target Setup

Predict short-horizon mid-price movement:

- move = (mid_price_change_1 != 0)
- direction = sign(mid_price_change_1) where move = 1

Final signal later:
P(up) = P(move) × P(up | move)
- Targets are defined using t+1 price 

In [13]:
final_dataset["mid_price_change_1_sign"].value_counts()

mid_price_change_1_sign
 0.0    34152
 1.0     6483
-1.0     6463
Name: count, dtype: int64

In [14]:
final_dataset["mid_price_moves"] = (final_dataset["mid_price_change_1"] != 0).astype(
    int
)

In [15]:
PATH = "C:\\Users\\jayod\\Documents\\Quant_Project\\microstructure-alpha-engine\\data\\ml_ready_data\\ml_ready.parquet"

final_dataset.to_parquet(
    PATH,
    compression="snappy",
)

- Dataset is strictly time-ordered
- All features computed using past information only